In [ ]:
import pandas as pd
import numpy as np
import os

In [44]:

# Configuration for output
OUTPUT_DIR = "../data/cleaned/"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def to_snake_case(df):
    """Standardizes column names to snake_case and removes whitespace."""
    df.columns = (df.columns
                  .str.strip()
                  .str.replace(' ', '_')
                  .str.replace('(', '')
                  .str.replace(')', '')
                  .str.lower())
    return df

In [45]:
shipping_raw = pd.read_csv("../data/raw/shipping_rates.csv")
port_raw = pd.read_csv("../data/raw/port_congestion.csv")
trade_raw = pd.read_csv("../data/raw/trade_flows.csv")
events_raw = pd.read_csv("../data/raw/disruption_events.csv")
industry_raw = pd.read_csv("../data/raw/industry_exposure.csv")

In [46]:
# Dictionary mapping for permanent reference
raw_data = {
    'shipping': shipping_raw,
    'port': port_raw,
    'trade': trade_raw,
    'events': events_raw,
    'industry': industry_raw
}

print("--- Running Global Standardization ---")
for name, df in raw_data.items():
    # Fix 1: Permanent name update
    raw_data[name] = to_snake_case(df)
    
    # Fix 2: Permanent string trimming
    obj_cols = raw_data[name].select_dtypes(include='object').columns
    raw_data[name][obj_cols] = raw_data[name][obj_cols].apply(lambda x: x.str.strip())
    
    print(f"Standardized {name}: {raw_data[name].shape[1]} columns processed.")

--- Running Global Standardization ---
Standardized shipping: 12 columns processed.
Standardized port: 11 columns processed.
Standardized trade: 11 columns processed.
Standardized events: 18 columns processed.
Standardized industry: 15 columns processed.


In [47]:
shipping_df = raw_data['shipping'].copy()
shipping_df['date'] = pd.to_datetime(shipping_df['date'])

# Prepare join keys for Dim_Date
shipping_df['year'] = shipping_df['date'].dt.year
shipping_df['month'] = shipping_df['date'].dt.month

shipping_df['rolling_6m_container'] = shipping_df['container_rate_usd_40ft'].rolling(window=6).mean()
shipping_df['is_baseline_year'] = (shipping_df['year'] == 2000).astype(int)

In [48]:
port_df = raw_data['port'].copy()
port_df['date'] = pd.to_datetime(port_df['week_start'])

# Fix: Standardize to 0-100 scale (Assuming raw was 0.0-1.0)
if port_df['port_utilization_pct'].max() <= 1.1:
    print("Standardizing Port Utilization from decimal to 0-100 scale.")
    port_df['port_utilization_pct'] = port_df['port_utilization_pct'] * 100

# Region Mapping
region_map = {
    'Shanghai': 'Asia', 'Singapore': 'Asia', 'Ningbo': 'Asia', 'Shenzhen': 'Asia', 
    'Busan': 'Asia', 'Hong Kong': 'Asia', 'Qingdao': 'Asia', 'Guangzhou': 'Asia',
    'Tanjung Pelepas': 'Asia', 'Laem Chabang': 'Asia', 'Tanjung Priok': 'Asia', 'Colombo': 'Asia',
    'Los Angeles': 'North America', 'Long Beach': 'North America', 'New York': 'North America',
    'Rotterdam': 'Europe', 'Hamburg': 'Europe', 'Antwerp': 'Europe', 'Felixstowe': 'Europe',
    'Dubai (Jebel Ali)': 'Middle East'
}
port_df['region'] = port_df['port'].map(region_map)

# Save region map as a seed for Dim_Geography later
pd.DataFrame(list(region_map.items()), columns=['port', 'region']).to_csv(f"{OUTPUT_DIR}geo_mapping_seed.csv", index=False)

Standardizing Port Utilization from decimal to 0-100 scale.


In [49]:
trade_df = raw_data['trade'].copy()

print(f"Trade rows BEFORE filtering: {len(trade_df)}")
trade_df = trade_df[trade_df['trade_active'] == 1].reset_index(drop=True)
print(f"Trade rows AFTER filtering: {len(trade_df)}")

# Boolean to SmallInt
trade_df['supply_chain_integrated'] = trade_df['supply_chain_integrated'].astype(int)

Trade rows BEFORE filtering: 1250
Trade rows AFTER filtering: 1225


In [50]:
events_df = events_raw.copy()
severity_rank = {'low': 1, 'medium': 2, 'high': 3, 'extreme': 4}
events_df['severity_score'] = events_df['severity'].map(severity_rank)


In [51]:
industry_df = industry_raw.copy()
# Ensure all vulnerability scores are on a consistent 0-10 scale
vuln_cols = [col for col in industry_df.columns if 'vulnerability' in col or 'exposure' in col]
for col in vuln_cols:
    industry_df[col] = industry_df[col].clip(0, 10)

In [52]:
print("--- Final Quality Gate Validation ---")

# 1. Null Checks
print(f"Shipping Nulls: {shipping_df.isnull().sum().sum()}")
print(f"Port Nulls: {port_df.isnull().sum().sum()}")

# 2. Logic Checks
assert port_df['port_utilization_pct'].max() <= 120, "Utilization normalization failed!"
assert trade_df['trade_active'].min() == 1, "Inactive trade flows were not removed!"

# 3. Statistical Profiling
print("--- Cleaned Layer Summary ---")
display(shipping_df.describe())

# 4. Data Type Audit
print("\n--- Port Data Types ---")
print(port_df.dtypes)

--- Final Quality Gate Validation ---
Shipping Nulls: 18
Port Nulls: 0
--- Cleaned Layer Summary ---


,date,year,month,baltic_dry_index,container_rate_usd_40ft,air_cargo_rate_usd_kg,bdi_mom_change_pct,container_yoy_pct,tanker_rate_aframax_usd_day,bulk_carrier_handysize_usd_day,supply_chain_pressure_index,on_time_delivery_pct,rolling_6m_container,is_baseline_year
count,300,300.000000,300.00000,300.000000,300.000000,300.00000,299.000000,288.000000,300.000000,300.000000,300.00000,300.000000,295.000000,300.000000
mean,2012-06-16 02:14:24,2012.000000,6.50000,2374.113333,2989.133333,3.95600,2.445987,8.233646,26444.463333,11155.036667,1.09880,0.898237,2988.570621,0.040000
min,2000-01-01 00:00:00,2000.000000,1.00000,300.000000,1570.000000,2.00000,-45.080000,-73.730000,8000.000000,3000.000000,0.50000,0.687000,1668.333333,0.000000
25%,2006-03-24 06:00:00,2006.000000,3.75000,1339.000000,2130.000000,3.38000,-5.935000,-8.685000,20642.000000,7926.250000,0.81000,0.888750,2152.500000,0.000000
50%,2012-06-16 00:00:00,2012.000000,6.50000,1823.500000,2390.000000,3.73500,0.000000,2.480000,25715.500000,11254.500000,0.99500,0.907000,2438.333333,0.000000
75%,2018-09-08 12:00:00,2018.000000,9.25000,2560.750000,2802.500000,4.14250,6.515000,18.252500,30747.750000,13945.750000,1.23000,0.923000,2715.833333,0.000000
max,2024-12-01 00:00:00,2024.000000,12.00000,9930.000000,12560.000000,8.79000,690.000000,193.540000,80000.000000,21829.000000,2.92000,0.980000,11166.666667,1.000000
std,NaN,7.223151,3.45782,1712.922795,1950.802746,0.98045,41.356688,36.709979,10599.641654,4472.017218,0.47657,0.044803,1912.641154,0.196287



--- Port Data Types ---
week_start                      object
year                             int64
port                            object
country                         object
region                          object
throughput_teu_mn                int64
vessels_at_anchor                int64
avg_wait_days                  float64
congestion_index               float64
port_utilization_pct           float64
berth_delay_hrs                float64
date                    datetime64[ns]
dtype: object


In [53]:
# Validation: Ensure nulls are ONLY in the baseline period
baseline_nulls = shipping_df[shipping_df['year'] == 2000]['container_yoy_pct'].isnull().sum()
other_nulls = shipping_df[shipping_df['year'] > 2000]['container_yoy_pct'].isnull().sum()

print(f"Baseline Nulls: {baseline_nulls}") # Expected: 12
print(f"Unexpected Nulls: {other_nulls}")   # Expected: 0

Baseline Nulls: 12
Unexpected Nulls: 0


In [54]:
shipping_df.to_csv(f"{OUTPUT_DIR}cleaned_shipping_rates.csv", index=False)
port_df.to_csv(f"{OUTPUT_DIR}cleaned_port_congestion.csv", index=False)
trade_df.to_csv(f"{OUTPUT_DIR}cleaned_trade_flows.csv", index=False)
events_df.to_csv(f"{OUTPUT_DIR}cleaned_disruption_events.csv", index=False)
industry_df.to_csv(f"{OUTPUT_DIR}cleaned_industry_exposure.csv", index=False) 

print(f"Cleaning Complete. {len(os.listdir(OUTPUT_DIR))} files exported to {OUTPUT_DIR}")

Cleaning Complete. 6 files exported to ../data/cleaned/
